# Fase 5 · M07: Comparativa Final

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 5 — Modelado Clásico |
| **Módulo** | M07 — Comparativa Final |

---

## 🎯 Qué hace

Carga los resultados de los 6 módulos (m01–m06), construye la tabla maestra de 21 algoritmos, genera gráficos comparativos y selecciona el top-3 para Fase 6.

## 📋 Requisitos

- `results/fase5/results_lineales_completo.parquet`
- `results/fase5/results_arboles.parquet`
- `results/fase5/results_boosting.parquet`
- `results/fase5/results_otros.parquet`
- `results/fase5/results_mlp_ebm.parquet`
- `results/fase5/results_ensambles.parquet`

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `results/fase5/resultados_maestro.parquet` | Tabla maestra 21 algoritmos |
| `results/fase5/resultados_maestro.xlsx` | Versión Excel |
| `results/fase5/top3_fase6.json` | Top-3 para Fase 6 |
| `docs/html/fase5/m07_comparacion.html` | Informe HTML |

## 🔄 Flujo

```
results/fase5/results_*.parquet (6 módulos)
    ↓ Unificación tabla maestra
    ↓ Ranking por AUC CV y F1
    ↓ Selección top-3 por familia
    → resultados_maestro.parquet + top3_fase6.json + HTML
```

## ➡️ Siguiente

Fin de Fase 5 — continuar con `f6_m00_preparacion.ipynb`


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN Y DEPENDENCIAS
# ============================================================================
import sys
from pathlib import Path

# ROOT robusto — sube niveles hasta encontrar src/
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from src.config import RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.utils.graficos import figura_a_base64
from src.html.render import render_pagina_desde_fichero

NOMBRE_FICHERO = 'f5_m07_comparacion.ipynb'
fmt = formato_numero_es

# ── Rutas ─────────────────────────────────────────────────────────────────────
RUTA_RESULTS  = ROOT / 'data' / '05_modelado' / 'results'
RUTA_FASE5_HTML = RUTA_HTML / 'fase5'
RUTA_FASE5_HTML.mkdir(parents=True, exist_ok=True)

# ── Colores por familia ────────────────────────────────────────────────────────
COLOR_FAMILIA = {
    'Lineales':          '#3182ce',
    'Árboles':           '#38a169',
    'Gradient Boosting': '#805ad5',
    'Otros':             '#ed8936',
    'MLP + EBM':         '#e53e3e',
    'Ensambles':         '#d69e2e',
}

# ── Baseline AutoML ───────────────────────────────────────────────────────────
BASELINE_F1  = 0.7970
BASELINE_AUC = 0.9300

graficos_b64 = {}  # acumulador de gráficos base64

info_entorno()
print(f'\n📂 RESULTS : {RUTA_RESULTS}')


✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓ ============================

In [2]:
# ============================================================================
# CELDA 2: CARGA Y UNIFICACIÓN DE PARQUETS
# ============================================================================
# Parquets definitivos por módulo. results_lineales_completo.parquet
# ya contiene los 7 modelos lineales (basico + ext combinados).
# Se excluyen los parquets de prueba y los parciales (basico, ext).

PARQUETS = {
    'Lineales':          'results_lineales_completo.parquet',
    'Árboles':           'results_arboles.parquet',
    'Gradient Boosting': 'results_boosting.parquet',
    'Otros':             'results_otros.parquet',
    'MLP + EBM':         'results_mlp_ebm.parquet',
    'Ensambles':         'results_ensambles.parquet',
}

dfs = []
for familia, fichero in PARQUETS.items():
    ruta = RUTA_RESULTS / fichero
    if not ruta.exists():
        print(f'⚠️  {fichero} — no encontrado, se omite')
        continue
    df = pd.read_parquet(ruta)
    df['familia'] = familia  # asegurar familia correcta
    dfs.append(df)
    print(f'✅ {fichero}: {len(df)} filas')

df_maestro = pd.concat(dfs, ignore_index=True)

# Columnas comunes (auc_pr_test solo en m05/m06 — rellenar con NaN)
COLS_BASE = ['familia', 'modelo', 'estrategia',
             'auc_mean', 'auc_std', 'f1_mean', 'f1_std',
             'precision_mean', 'recall_mean', 'tiempo_s',
             'auc_test', 'f1_test', 'precision_test', 'recall_test',
             'kappa_test', 'logloss_test']

for col in COLS_BASE:
    if col not in df_maestro.columns:
        df_maestro[col] = np.nan

df_maestro = df_maestro[COLS_BASE].copy()
df_maestro = df_maestro.sort_values('auc_mean', ascending=False).reset_index(drop=True)

print(f'\n📊 Tabla maestra: {df_maestro.shape[0]} combinaciones × {df_maestro.shape[1]} columnas')
print(f'   Familias: {df_maestro["familia"].nunique()}')
print(f'   Modelos únicos: {df_maestro["modelo"].nunique()}')


✅ results_lineales_completo.parquet: 21 filas
✅ results_arboles.parquet: 9 filas
✅ results_boosting.parquet: 12 filas
✅ results_otros.parquet: 9 filas
✅ results_mlp_ebm.parquet: 6 filas
✅ results_ensambles.parquet: 12 filas

📊 Tabla maestra: 69 combinaciones × 16 columnas
   Familias: 6
   Modelos únicos: 23


In [3]:
# ============================================================================
# CELDA 3: MEJOR POR FAMILIA
# ============================================================================
# Para cada familia, seleccionar la combinación con mayor AUC CV.

df_mejores = (
    df_maestro
    .sort_values('auc_mean', ascending=False)
    .groupby('familia', sort=False)
    .first()
    .reset_index()
    .sort_values('auc_mean', ascending=False)
)

print('🏆 MEJOR COMBINACIÓN POR FAMILIA')
print('=' * 70)
for _, row in df_mejores.iterrows():
    supera = '✅ SUPERA BASELINE' if row['f1_mean'] >= BASELINE_F1 else '  '
    print(f"  {row['familia']:20s} {row['modelo']:15s} {row['estrategia']:10s} "
          f"AUC={row['auc_mean']:.4f}  F1={row['f1_mean']:.4f}  {supera}")
print(f'\n  Baseline AutoML: AUC={BASELINE_AUC:.4f}  F1={BASELINE_F1:.4f}')


🏆 MEJOR COMBINACIÓN POR FAMILIA
  Gradient Boosting    XGBoost         none       AUC=0.9318  F1=0.7862    
  Ensambles            Stacking        balanced   AUC=0.9308  F1=0.7882    
  Árboles              RandomForest    none       AUC=0.9256  F1=0.7580    
  MLP + EBM            EBM             none       AUC=0.9202  F1=0.7672    
  Lineales             SVM_RBF         balanced   AUC=0.9056  F1=0.7565    
  Otros                KNN             none       AUC=0.8827  F1=0.6997    

  Baseline AutoML: AUC=0.9300  F1=0.7970


In [4]:
# ============================================================================
# CELDA 3b: TABLA RESUMEN POR MÓDULO (para HTML y consola)
# ============================================================================
# Reproduce la tabla del chat: mejor modelo por módulo con AUC y F1 CV.

MODULOS_INFO = [
    {'modulo': 'm01', 'nombre': 'Lineales',          'parquet': 'results_lineales_completo.parquet'},
    {'modulo': 'm02', 'nombre': 'Árboles',            'parquet': 'results_arboles.parquet'},
    {'modulo': 'm03', 'nombre': 'Gradient Boosting',  'parquet': 'results_boosting.parquet'},
    {'modulo': 'm04', 'nombre': 'Otros',              'parquet': 'results_otros.parquet'},
    {'modulo': 'm05', 'nombre': 'MLP + EBM',          'parquet': 'results_mlp_ebm.parquet'},
    {'modulo': 'm06', 'nombre': 'Ensambles',          'parquet': 'results_ensambles.parquet'},
]

resumen_modulos = []
for info in MODULOS_INFO:
    ruta = RUTA_RESULTS / info['parquet']
    if not ruta.exists():
        continue
    df_mod = pd.read_parquet(ruta).sort_values('auc_mean', ascending=False)
    mejor = df_mod.iloc[0]
    resumen_modulos.append({
        'Módulo':     info['modulo'].upper(),
        'Familia':    info['nombre'],
        'Modelo':     mejor['modelo'],
        'Estrategia': mejor['estrategia'],
        'AUC CV':     f"{mejor['auc_mean']:.4f} ± {mejor['auc_std']:.4f}",
        'F1 CV':      f"{mejor['f1_mean']:.4f}",
        'Precision':  f"{mejor['precision_mean']:.4f}",
        'Recall':     f"{mejor['recall_mean']:.4f}",
    })

df_resumen = pd.DataFrame(resumen_modulos)

print('RESUMEN POR MÓDULO — mejor combinación de cada familia')
print('=' * 80)
print(df_resumen.to_string(index=False))
print(f'\n  Baseline AutoML: AUC≈{BASELINE_AUC:.4f}  F1={BASELINE_F1:.4f}')


RESUMEN POR MÓDULO — mejor combinación de cada familia
Módulo           Familia       Modelo Estrategia          AUC CV  F1 CV Precision Recall
   M01          Lineales      SVM_RBF   balanced 0.9056 ± 0.0034 0.7565    0.7120 0.8071
   M02           Árboles RandomForest       none 0.9256 ± 0.0038 0.7580    0.8572 0.6796
   M03 Gradient Boosting      XGBoost       none 0.9318 ± 0.0032 0.7862    0.8485 0.7326
   M04             Otros          KNN       none 0.8827 ± 0.0021 0.6997    0.7841 0.6320
   M05         MLP + EBM          EBM       none 0.9202 ± 0.0036 0.7672    0.8341 0.7105
   M06         Ensambles     Stacking       none 0.9308 ± 0.0034 0.7882    0.8306 0.7502

  Baseline AutoML: AUC≈0.9300  F1=0.7970


In [5]:
# ============================================================================
# CELDA 4: SCATTER AUC CV vs F1 CV — mejor por familia
# ============================================================================

fig, ax = plt.subplots(figsize=(10, 7))

for _, row in df_mejores.iterrows():
    color = COLOR_FAMILIA.get(row['familia'], '#888888')
    ax.scatter(row['auc_mean'], row['f1_mean'],
               color=color, s=180, zorder=5, edgecolors='white', linewidths=1.5)
    ax.annotate(
        f"{row['modelo']}\n({row['familia']})",
        (row['auc_mean'], row['f1_mean']),
        textcoords='offset points', xytext=(8, 4),
        fontsize=8, color=color
    )

# Líneas de baseline
ax.axvline(BASELINE_AUC, color='#e53e3e', linestyle='--', alpha=0.6, linewidth=1.2,
           label=f'Baseline AUC={BASELINE_AUC}')
ax.axhline(BASELINE_F1,  color='#e53e3e', linestyle=':',  alpha=0.6, linewidth=1.2,
           label=f'Baseline F1={BASELINE_F1}')

# Zona "supera baseline"
ax.fill_between([BASELINE_AUC, 1.0], BASELINE_F1, 1.0,
                alpha=0.06, color='#38a169', label='Supera baseline')

ax.set_xlabel('AUC CV (mean)', fontsize=12)
ax.set_ylabel('F1 CV (mean)', fontsize=12)
ax.set_title('Comparativa Final — Mejor modelo por familia\nAUC vs F1 (5-Fold CV)', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
graficos_b64['scatter_familias'] = figura_a_base64(fig)
plt.close(fig)
print('✅ Scatter AUC vs F1 generado')


✅ Scatter AUC vs F1 generado


In [6]:
# ============================================================================
# CELDA 5: RANKING TOP-10 — barras horizontales AUC CV
# ============================================================================

top10 = df_maestro.head(10).copy()
top10['etiqueta'] = top10['modelo'] + ' (' + top10['estrategia'] + ')'

fig, ax = plt.subplots(figsize=(10, 6))
colores = [COLOR_FAMILIA.get(f, '#888') for f in top10['familia']]
bars = ax.barh(range(len(top10)), top10['auc_mean'], color=colores,
               alpha=0.85, edgecolor='white')

# Línea baseline
ax.axvline(BASELINE_AUC, color='#e53e3e', linestyle='--', linewidth=1.5,
           label=f'Baseline AutoML AUC={BASELINE_AUC}', zorder=5)

for i, (_, row) in enumerate(top10.iterrows()):
    ax.text(row['auc_mean'] + 0.001, i,
            f"  {row['auc_mean']:.4f} ± {row['auc_std']:.4f}",
            va='center', fontsize=8.5)

ax.set_yticks(range(len(top10)))
ax.set_yticklabels(top10['etiqueta'], fontsize=9)
ax.set_xlabel('AUC CV (mean)', fontsize=11)
ax.set_title('Top-10 combinaciones — AUC CV', fontsize=13)
ax.set_xlim(top10['auc_mean'].min() - 0.01, top10['auc_mean'].max() + 0.03)
ax.legend(fontsize=9)
ax.invert_yaxis()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Leyenda familias
patches = [mpatches.Patch(color=c, label=f) for f, c in COLOR_FAMILIA.items()
           if f in top10['familia'].values]
ax.legend(handles=patches + [
    plt.Line2D([0], [0], color='#e53e3e', linestyle='--', label=f'Baseline AUC={BASELINE_AUC}')
], fontsize=8, loc='lower right')

plt.tight_layout()
graficos_b64['ranking_top10'] = figura_a_base64(fig)
plt.close(fig)
print('✅ Ranking top-10 generado')


✅ Ranking top-10 generado


In [7]:
# ============================================================================
# CELDA 6: COMPARATIVA POR FAMILIA — barras AUC y F1
# ============================================================================

familias = df_mejores['familia'].tolist()
aucs = df_mejores['auc_mean'].tolist()
f1s  = df_mejores['f1_mean'].tolist()
colores_fam = [COLOR_FAMILIA.get(f, '#888') for f in familias]

x = np.arange(len(familias))
w = 0.35

fig, ax = plt.subplots(figsize=(11, 6))
b1 = ax.bar(x - w/2, aucs, w, label='AUC CV', color=colores_fam, alpha=0.85)
b2 = ax.bar(x + w/2, f1s,  w, label='F1 CV',  color=colores_fam, alpha=0.50,
            edgecolor=[COLOR_FAMILIA.get(f,'#888') for f in familias], linewidth=1.5)

ax.axhline(BASELINE_AUC, color='#e53e3e', linestyle='--', linewidth=1.2, alpha=0.7,
           label=f'Baseline AUC={BASELINE_AUC}')
ax.axhline(BASELINE_F1,  color='#e53e3e', linestyle=':',  linewidth=1.2, alpha=0.7,
           label=f'Baseline F1={BASELINE_F1}')

for bar in b1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar in b2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(familias, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Métrica CV (mean)', fontsize=11)
ax.set_title('Mejor modelo por familia — AUC y F1 CV', fontsize=13)
ax.set_ylim(0.80, max(aucs + f1s) + 0.03)
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
graficos_b64['comparativa_familias'] = figura_a_base64(fig)
plt.close(fig)
print('✅ Comparativa por familia generada')


✅ Comparativa por familia generada


In [8]:
# ============================================================================
# CELDA 7: SELECCIÓN TOP-3 PARA FASE 6
# ============================================================================
# Criterios:
#   1. Máximo AUC CV (rendimiento)
#   2. Diversidad de familia (no duplicar familia en top-3)
#   3. Interpretabilidad viable (EBM preferido sobre black-box puro)

# Top por AUC CV, 1 por familia
top3_candidatos = (
    df_maestro
    .sort_values('auc_mean', ascending=False)
    .groupby('familia', sort=False)
    .first()
    .reset_index()
    .sort_values('auc_mean', ascending=False)
    .head(3)
)

print('🏆 TOP-3 SELECCIONADOS PARA FASE 6')
print('=' * 70)
for i, (_, row) in enumerate(top3_candidatos.iterrows(), 1):
    supera = '✅' if row['f1_mean'] >= BASELINE_F1 else '⚠️'
    print(f"  #{i} {row['modelo']:15s} ({row['familia']:20s}) "
          f"AUC={row['auc_mean']:.4f}  F1={row['f1_mean']:.4f}  {supera}")

TOP3_MODELOS = top3_candidatos[['familia','modelo','estrategia','auc_mean','f1_mean']].to_dict('records')

# Visualización top-3
fig, ax = plt.subplots(figsize=(8, 4))
nombres = [f"{r['modelo']}\n({r['familia']})" for r in TOP3_MODELOS]
aucs_top3 = [r['auc_mean'] for r in TOP3_MODELOS]
colores_top3 = [COLOR_FAMILIA.get(r['familia'], '#888') for r in TOP3_MODELOS]

bars = ax.bar(nombres, aucs_top3, color=colores_top3, alpha=0.85, edgecolor='white', width=0.5)
ax.axhline(BASELINE_AUC, color='#e53e3e', linestyle='--', linewidth=1.5,
           label=f'Baseline AutoML AUC={BASELINE_AUC}')

for bar, val in zip(bars, aucs_top3):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('AUC CV (mean)', fontsize=11)
ax.set_title('Top-3 para Fase 6 — Interpretabilidad + Fairness', fontsize=12)
ax.set_ylim(min(aucs_top3) - 0.02, max(aucs_top3) + 0.02)
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
graficos_b64['top3'] = figura_a_base64(fig)
plt.close(fig)
print('\n✅ Top-3 generado')


🏆 TOP-3 SELECCIONADOS PARA FASE 6
  #1 XGBoost         (Gradient Boosting   ) AUC=0.9318  F1=0.7862  ⚠️
  #2 Stacking        (Ensambles           ) AUC=0.9308  F1=0.7882  ⚠️
  #3 RandomForest    (Árboles             ) AUC=0.9256  F1=0.7580  ⚠️

✅ Top-3 generado


In [9]:
# ============================================================================
# CELDA 8: SERIALIZACIÓN — resultados_maestro
# ============================================================================
import json as json_lib

ruta_parquet = RUTA_RESULTS / 'resultados_maestro.parquet'
ruta_xlsx    = RUTA_RESULTS / 'resultados_maestro.xlsx'
ruta_json    = RUTA_RESULTS / 'resultados_maestro.json'

df_maestro.to_parquet(ruta_parquet, index=False)
df_maestro.to_excel(ruta_xlsx, index=False)
df_maestro.to_json(ruta_json, orient='records', force_ascii=False, indent=2)

# Guardar top-3
ruta_top3 = RUTA_RESULTS / 'top3_fase6.json'
with open(ruta_top3, 'w', encoding='utf-8') as f:
    json_lib.dump(TOP3_MODELOS, f, ensure_ascii=False, indent=2)

print(f'✅ resultados_maestro.parquet — {len(df_maestro)} filas')
print(f'✅ resultados_maestro.xlsx')
print(f'✅ resultados_maestro.json')
print(f'✅ top3_fase6.json — {len(TOP3_MODELOS)} modelos')


✅ resultados_maestro.parquet — 69 filas
✅ resultados_maestro.xlsx
✅ resultados_maestro.json
✅ top3_fase6.json — 3 modelos


In [10]:
# ============================================================================
# CELDA 9: GENERAR HTML — m07_comparacion.html
# ============================================================================
from src.html import generar_kpis_html, generar_seccion_html, guardar_html

# ── KPIs ──────────────────────────────────────────────────────────────────────
n_combinaciones = len(df_maestro)
n_modelos       = df_maestro['modelo'].nunique()
mejor_auc       = df_maestro['auc_mean'].max()
mejor_f1        = df_maestro['f1_mean'].max()

KPIS = [
    {'valor': str(n_combinaciones),    'titulo': 'Combinaciones',  'color': '#3182ce'},
    {'valor': str(n_modelos),          'titulo': 'Modelos únicos', 'color': '#38a169'},
    {'valor': f'{mejor_auc:.4f}',      'titulo': 'Mejor AUC CV',   'color': '#805ad5'},
    {'valor': f'{mejor_f1:.4f}',       'titulo': 'Mejor F1 CV',    'color': '#e53e3e'},
]

# ── Tabla maestra (top-20 por AUC) ───────────────────────────────────────────
def fila_tabla(row, i):
    bg = '#ebf8ff' if i == 0 else ('#f7fafc' if i % 2 == 0 else 'white')
    supera = '✅' if row['f1_mean'] >= BASELINE_F1 else ''
    return (
        f'<tr style="background:{bg}">'
        f'<td>{i+1}</td>'
        f'<td><strong>{row["modelo"]}</strong></td>'
        f'<td>{row["familia"]}</td>'
        f'<td>{row["estrategia"]}</td>'
        f'<td><strong>{row["auc_mean"]:.4f}</strong> ± {row["auc_std"]:.4f}</td>'
        f'<td>{row["f1_mean"]:.4f}</td>'
        f'<td>{row["precision_mean"]:.4f}</td>'
        f'<td>{row["recall_mean"]:.4f}</td>'
        f'<td>{supera}</td>'
        f'</tr>'
    )

filas = ''.join(fila_tabla(row, i) for i, (_, row) in enumerate(df_maestro.head(20).iterrows()))
tabla_html = (
    '<table class="tabla-datos"><thead><tr>'
    '<th>#</th><th>Modelo</th><th>Familia</th><th>Estrategia</th>'
    '<th>AUC CV</th><th>F1 CV</th><th>Precision</th><th>Recall</th><th>Baseline</th>'
    '</tr></thead><tbody>' + filas + '</tbody></table>'
    f'<p style="color:#666;font-size:0.9em">Mostrando top-20 de {n_combinaciones} combinaciones — '
    f'ordenadas por AUC CV</p>'
)

# ── Top-3 cards ───────────────────────────────────────────────────────────────
def card_top3(r, pos):
    medalla = ['🥇','🥈','🥉'][pos]
    color = COLOR_FAMILIA.get(r['familia'], '#3182ce')
    return (
        f'<div style="background:#f7fafc;border:2px solid {color};border-radius:12px;'
        f'padding:20px;text-align:center;min-width:180px;flex:1">'
        f'<div style="font-size:2rem">{medalla}</div>'
        f'<div style="font-size:1.3rem;font-weight:700;color:{color};margin:8px 0">{r["modelo"]}</div>'
        f'<div style="color:#555;font-size:0.9rem">{r["familia"]}</div>'
        f'<div style="margin-top:12px"><strong>AUC CV</strong>: {r["auc_mean"]:.4f}</div>'
        f'<div><strong>F1 CV</strong>: {r["f1_mean"]:.4f}</div>'
        f'<div style="color:#666;font-size:0.85rem;margin-top:4px">Estrategia: {r["estrategia"]}</div>'
        f'</div>'
    )

cards_html = '<div style="display:flex;flex-wrap:wrap;gap:16px;margin:16px 0">' + \
    ''.join(card_top3(r, i) for i, r in enumerate(TOP3_MODELOS)) + '</div>'

# ── Conclusiones ──────────────────────────────────────────────────────────────
mejor = TOP3_MODELOS[0]
conclusiones_html = (
    f'<p>De las <strong>{n_combinaciones} combinaciones</strong> evaluadas sobre '
    f'{fmt(26896)} expedientes universitarios (5-Fold Stratified CV), '
    f'<strong>{mejor["modelo"]}</strong> ({mejor["familia"]}) '
    f'con estrategia <strong>{mejor["estrategia"]}</strong> obtiene el mejor resultado '
    f'con <strong>AUC CV = {mejor["auc_mean"]:.4f}</strong> y '
    f'<strong>F1 CV = {mejor["f1_mean"]:.4f}</strong>.</p>'
    f'<p>El baseline AutoML (CatBoost_BAG_L2, F1=0.7970, AUC≈0.93) '
    f'{"ha sido superado" if mejor["f1_mean"] >= BASELINE_F1 else "no ha sido superado"} '
    f'en F1 por el modelado manual controlado.</p>'
    f'<p>Los <strong>3 modelos seleccionados para Fase 6</strong> (interpretabilidad + fairness) '
    f'representan familias diversas y ofrecen distintos perfiles precision/recall, '
    f'permitiendo un análisis robusto de explicabilidad y sesgo.</p>'
)

# ── Ensamblar ─────────────────────────────────────────────────────────────────
# ── Tabla resumen por módulo ──────────────────────────────────────────────────
def fila_resumen(row, i):
    bg = '#ebf8ff' if i == 0 else ('#f7fafc' if i % 2 == 0 else 'white')
    es_mejor = row['AUC CV'] == df_resumen['AUC CV'].max()
    return (
        f'<tr style="background:{bg}{";\"font-weight:600\"" if es_mejor else ""}">'
        f'<td><strong>{row["Módulo"]}</strong></td>'
        f'<td>{row["Familia"]}</td>'
        f'<td><strong>{row["Modelo"]}</strong></td>'
        f'<td>{row["Estrategia"]}</td>'
        f'<td><strong>{row["AUC CV"]}</strong></td>'
        f'<td>{row["F1 CV"]}</td>'
        f'<td>{row["Precision"]}</td>'
        f'<td>{row["Recall"]}</td>'
        f'</tr>'
    )

filas_resumen = ''.join(fila_resumen(row, i) for i, (_, row) in enumerate(df_resumen.iterrows()))
tabla_resumen_html = (
    '<table class="tabla-datos"><thead><tr>'
    '<th>Módulo</th><th>Familia</th><th>Mejor modelo</th><th>Estrategia</th>'
    '<th>AUC CV</th><th>F1 CV</th><th>Precision</th><th>Recall</th>'
    '</tr></thead><tbody>' + filas_resumen + '</tbody></table>'
    f'<p style="color:#666;font-size:0.9em">Baseline AutoML: '
    f'AUC≈{BASELINE_AUC:.4f} · F1={BASELINE_F1:.4f}</p>'
)

contenido = (
    generar_kpis_html(KPIS)
    + generar_seccion_html('Resumen por módulo — mejor combinación', tabla_resumen_html)
    + generar_seccion_html('Comparativa AUC vs F1 — mejor por familia',
        f'<img src="data:image/png;base64,{graficos_b64.get("scatter_familias","")}" style="max-width:100%">')
    + generar_seccion_html('Top-10 modelos por AUC CV',
        f'<img src="data:image/png;base64,{graficos_b64.get("ranking_top10","")}" style="max-width:100%">')
    + generar_seccion_html('Comparativa por familia',
        f'<img src="data:image/png;base64,{graficos_b64.get("comparativa_familias","")}" style="max-width:100%">')
    + generar_seccion_html('Top-20 combinaciones — tabla maestra', tabla_html)
    + generar_seccion_html('Selección Top-3 para Fase 6',
        cards_html +
        f'<img src="data:image/png;base64,{graficos_b64.get("top3","")}" style="max-width:100%">')
    + generar_seccion_html('Conclusiones', conclusiones_html)
)

html = render_pagina_desde_fichero(
    nombre_fichero=NOMBRE_FICHERO,
    contenido=contenido,
    carpeta_notebook='fase5_modelado'
)

ruta_html_out = RUTA_FASE5_HTML / 'm07_comparacion.html'
guardar_html(html, ruta_html_out)
print(f'✅ HTML generado: {ruta_html_out}')


SyntaxError: f-string expression part cannot include a backslash (3640542081.py, line 99)

In [ ]:
# ============================================================================
# CELDA 10: RESUMEN FINAL
# ============================================================================
print('=' * 60)
print('RESUMEN FINAL — f5_m07_comparacion')
print('=' * 60)
print(f'Combinaciones evaluadas : {n_combinaciones}')
print(f'Modelos únicos          : {n_modelos}')
print(f'Familias                : {df_maestro["familia"].nunique()}')
print()
print(f'🏆 Mejor global: {df_maestro.iloc[0]["modelo"]} ({df_maestro.iloc[0]["familia"]})')
print(f'   AUC CV = {df_maestro.iloc[0]["auc_mean"]:.4f} ± {df_maestro.iloc[0]["auc_std"]:.4f}')
print(f'   F1  CV = {df_maestro.iloc[0]["f1_mean"]:.4f}')
print()
print('Top-3 para Fase 6:')
for i, r in enumerate(TOP3_MODELOS, 1):
    print(f'  #{i} {r["modelo"]:15s} ({r["familia"]:20s}) AUC={r["auc_mean"]:.4f}  F1={r["f1_mean"]:.4f}')
print()
print('📁 Archivos:')
print('   data/05_modelado/results/resultados_maestro.parquet')
print('   data/05_modelado/results/resultados_maestro.xlsx')
print('   data/05_modelado/results/resultados_maestro.json')
print('   data/05_modelado/results/top3_fase6.json')
print('   docs/html/fase5/m07_comparacion.html')
print()
print('➡️  Siguiente: f6_m01_shap.ipynb (Fase 6 — Interpretabilidad)')
